# Capacity Scenario Analysis: Box Plot Visualizations

Evaluates the impact of marginal capacity increases (105%–120% of baseline) across four provider/diagnostic categories:

| Scenario Group | Parameters Varied | Baseline |
|---|---|---|
| **GYN Provider** | cytology · HPV-alone · co-test slots | 4 slots/day each |
| **PCP Throughput** | daily provider visit cap (`DAILY_PATIENTS`) | 2 slots/day (sim scale = 200 real/day) |
| **LDCT** | low-dose CT slots | 4 slots/day |
| **Cervical Biopsy** | colposcopy slots | 4 slots/day |

Each scenario runs **N_SEEDS** independent 80-year Monte Carlo simulations.  
Box plots show IQR with **5th–95th percentile** whiskers across seeds.

In [5]:
import sys
import re
import time
import colorsys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
})

# ── path setup ────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent.parent
for _sub in ['src', 'ModelParameters']:
    _p = str(PROJECT_ROOT / _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)

import parameters as cfg
from scenarios import ScenarioConfig
from mc_scenarios import run_mc_scenario

print(f'Project root : {PROJECT_ROOT}')
print(f'DAILY_PATIENTS: {cfg.DAILY_PATIENTS}')
print(f'CAPACITIES   : {cfg.CAPACITIES}')

Project root : /Users/Sophia/NYP
DAILY_PATIENTS: 2
CAPACITIES   : {'cytology': 4, 'hpv_alone': 4, 'co_test': 4, 'ldct': 4, 'colposcopy': 4, 'lung_biopsy': 4, 'leep': 4, 'cone_biopsy': 4}


In [ ]:
# ── simulation settings ───────────────────────────────────────────────────
N_SEEDS      = 50      # seeds per new scenario
N_WORKERS    = 3       # capped to avoid memory pressure on 8 GB RAM (7 workers → swap)
SEED_START   = 42
WARMUP_YEARS = cfg.WARMUP_YEARS   # 10 — exclude first 10 years from steady-state metrics

# capacity multipliers to evaluate (relative to current baseline)
MULTIPLIERS = [1.05, 1.10, 1.15, 1.20]

# output paths
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'Scenario Analysis Visualizations' / 'Capacity Scenario'
DATA_DIR   = PROJECT_ROOT / 'src' / 'mc_scenario_data' / 'capacity_scenarios'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# snapshot baseline values before any overrides
_BASE_CAP = dict(cfg.CAPACITIES)   # e.g. cytology=4, ldct=4, …
_BASE_DP  = cfg.DAILY_PATIENTS     # 2 (sim scale; × 100 = 200 real patients/day)

print(f'N_SEEDS      : {N_SEEDS}')
print(f'N_WORKERS    : {N_WORKERS}')
print(f'WARMUP_YEARS : {WARMUP_YEARS}')
print(f'MULTIPLIERS  : {MULTIPLIERS}')

In [7]:
# ── scenario group definitions ────────────────────────────────────────────
# Each group specifies which cfg parameters to scale and display metadata.
# Dotted-path keys  "CAPACITIES.<proc>"  →  cfg.CAPACITIES[<proc>]
# Top-level keys    "DAILY_PATIENTS"      →  cfg.DAILY_PATIENTS

SCENARIO_GROUPS = {
    'GYN': {
        'label':    'GYN Provider Capacity',
        'subtitle': 'Primary cervical screening slots (cytology · HPV-alone · co-test)',
        'color':    '#C2185B',   # deep pink
        'params':   {
            'CAPACITIES.cytology':  _BASE_CAP['cytology'],    # 4/day
            'CAPACITIES.hpv_alone': _BASE_CAP['hpv_alone'],   # 4/day
            'CAPACITIES.co_test':   _BASE_CAP['co_test'],     # 4/day
        },
    },
    'PCP': {
        'label':    'PCP Throughput',
        'subtitle': 'Daily provider visit capacity — all outpatient providers (PCP + GYN)',
        'color':    '#1565C0',   # dark blue
        'params':   {
            'DAILY_PATIENTS': _BASE_DP,   # 2/day (sim scale)
        },
    },
    'LDCT': {
        'label':    'LDCT Capacity',
        'subtitle': 'Daily low-dose CT slots for lung cancer screening',
        'color':    '#E65100',   # deep orange
        'params':   {
            'CAPACITIES.ldct': _BASE_CAP['ldct'],   # 4/day
        },
    },
    'Biopsy': {
        'label':    'Cervical Biopsy Capacity',
        'subtitle': 'Daily colposcopy slots for cervical cancer follow-up diagnostics',
        'color':    '#2E7D32',   # dark green
        'params':   {
            'CAPACITIES.colposcopy': _BASE_CAP['colposcopy'],   # 4/day
        },
    },
}

print('Scenario groups and capacity increments:')
for gk, g in SCENARIO_GROUPS.items():
    print(f'\n{gk} — {g["label"]}')
    for param, base in g['params'].items():
        vals = '  /  '.join(f'{base*m:.2f}' for m in MULTIPLIERS)
        print(f'  {param}: baseline={base}  →  {vals}')

Scenario groups and capacity increments:

GYN — GYN Provider Capacity
  CAPACITIES.cytology: baseline=4  →  4.20  /  4.40  /  4.60  /  4.80
  CAPACITIES.hpv_alone: baseline=4  →  4.20  /  4.40  /  4.60  /  4.80
  CAPACITIES.co_test: baseline=4  →  4.20  /  4.40  /  4.60  /  4.80

PCP — PCP Throughput
  DAILY_PATIENTS: baseline=2  →  2.10  /  2.20  /  2.30  /  2.40

LDCT — LDCT Capacity
  CAPACITIES.ldct: baseline=4  →  4.20  /  4.40  /  4.60  /  4.80

Biopsy — Cervical Biopsy Capacity
  CAPACITIES.colposcopy: baseline=4  →  4.20  /  4.40  /  4.60  /  4.80


In [8]:
# ── build ScenarioConfig objects (4 groups × 4 multipliers = 16 scenarios) ─
scenarios = {}   # name → (ScenarioConfig, group_key, multiplier)

for group_key, group in SCENARIO_GROUPS.items():
    for mult in MULTIPLIERS:
        pct  = int(round(mult * 100))
        name = f'{group_key.lower()}_{pct}pct'
        overrides = {
            param: base_val * mult
            for param, base_val in group['params'].items()
        }
        sc = ScenarioConfig(
            name=name,
            description=f'{group["label"]} at {pct}% of baseline',
            param_overrides=overrides,
            cotesting_enabled=False,
        )
        scenarios[name] = (sc, group_key, mult)

print(f'Total scenarios: {len(scenarios)}')
for name, (sc, gk, m) in scenarios.items():
    print(f'  {name:28s}  overrides = {sc.param_overrides}')

Total scenarios: 16
  gyn_105pct                    overrides = {'CAPACITIES.cytology': 4.2, 'CAPACITIES.hpv_alone': 4.2, 'CAPACITIES.co_test': 4.2}
  gyn_110pct                    overrides = {'CAPACITIES.cytology': 4.4, 'CAPACITIES.hpv_alone': 4.4, 'CAPACITIES.co_test': 4.4}
  gyn_115pct                    overrides = {'CAPACITIES.cytology': 4.6, 'CAPACITIES.hpv_alone': 4.6, 'CAPACITIES.co_test': 4.6}
  gyn_120pct                    overrides = {'CAPACITIES.cytology': 4.8, 'CAPACITIES.hpv_alone': 4.8, 'CAPACITIES.co_test': 4.8}
  pcp_105pct                    overrides = {'DAILY_PATIENTS': 2.1}
  pcp_110pct                    overrides = {'DAILY_PATIENTS': 2.2}
  pcp_115pct                    overrides = {'DAILY_PATIENTS': 2.3}
  pcp_120pct                    overrides = {'DAILY_PATIENTS': 2.4}
  ldct_105pct                   overrides = {'CAPACITIES.ldct': 4.2}
  ldct_110pct                   overrides = {'CAPACITIES.ldct': 4.4}
  ldct_115pct                   overrides = {'CAPACITI

## Run / Load Simulations

Results are **cached** to CSV. Re-running this cell skips any completed scenarios.

⏱ With `N_WORKERS = 3` on 8 GB RAM: ~15–20 min per new scenario, ~30–40 min for the 6 uncached ones  
✓ 10 scenarios already cached from prior runs — those are **instant**

In [ ]:
csv_paths = {}   # name → Path

for name, (sc, group_key, mult) in scenarios.items():
    csv_path = DATA_DIR / f'{name}_n{N_SEEDS}_start{SEED_START}.csv'
    if csv_path.exists():
        print(f'[CACHED]  {name}')
    else:
        t0 = time.time()
        print(f'[RUNNING] {name} …', end=' ', flush=True)
        run_mc_scenario(
            sc,
            n_seeds=N_SEEDS,
            seed_start=SEED_START,
            n_workers=N_WORKERS,
            out_csv=str(csv_path),
            progress=False,
        )
        elapsed = (time.time() - t0) / 60
        print(f'done ({elapsed:.1f} min)')
    csv_paths[name] = csv_path

print('\nAll scenarios ready.')

In [ ]:
# ── locate baseline CSV (highest n available) ─────────────────────────────
def _parse_n(p: Path) -> int:
    m = re.search(r'_n(\d+)_', p.stem)
    return int(m.group(1)) if m else 0

_bl_dir = PROJECT_ROOT / 'src' / 'mc_baseline_data'
_candidates = sorted(_bl_dir.glob('*baseline*_start42.csv'), key=_parse_n)
if not _candidates:
    # fall back: any CSV in the directory
    _candidates = sorted(_bl_dir.glob('*.csv'), key=_parse_n)
if not _candidates:
    raise FileNotFoundError(
        f'No baseline CSV found in {_bl_dir}. '
        'Run the baseline Monte Carlo simulation first.'
    )
BASELINE_CSV = _candidates[-1]
print(f'Using baseline: {BASELINE_CSV.name}  ({_parse_n(BASELINE_CSV)} seeds)')

In [ ]:
# ── metrics to display ────────────────────────────────────────────────────
# (metric_key, display_label, unit)
PLOT_METRICS = [
    ('revenue_capture_rate_pct',    'Revenue Capture Rate',      '%'),
    ('population_capture_rate_pct', 'Population Capture Rate',   '%'),
    ('mean_wait_primary_days',      'Primary Screening Wait',    'days'),
    ('mean_wait_secondary_days',    'Secondary Procedure Wait',  'days'),
    ('ltfu_rate_primary_pct',       'Primary LTFU Rate',         '%'),
    ('ltfu_rate_secondary_pct',     'Secondary LTFU Rate',       '%'),
]


def load_per_seed(csv_path: Path, warmup: int = 10) -> pd.DataFrame:
    """Return a DataFrame (index=seed) with one column per metric.

    Time-series metrics (multiple years): averaged over steady-state years (≥ warmup).
    Scalar metrics (single value per seed): taken as-is.
    """
    df = pd.read_csv(csv_path)
    records = []
    for seed, sdf in df.groupby('seed'):
        row = {'seed': seed}
        for metric_key, *_ in PLOT_METRICS:
            sub = sdf[sdf['metric'] == metric_key]
            if sub.empty:
                row[metric_key] = np.nan
                continue
            if sub['year'].nunique() > 1:
                sub = sub[sub['year'] >= warmup]
            row[metric_key] = sub['value'].mean()
        records.append(row)
    return pd.DataFrame(records).set_index('seed')


print('Loading baseline …', end=' ', flush=True)
baseline_df = load_per_seed(BASELINE_CSV, warmup=WARMUP_YEARS)
print(f'{len(baseline_df)} seeds loaded')

scenario_dfs = {}
for name, csv_path in csv_paths.items():
    scenario_dfs[name] = load_per_seed(csv_path, warmup=WARMUP_YEARS)

print(f'Scenario datasets loaded: {len(scenario_dfs)}')
baseline_df.describe().round(2)

In [ ]:
# ── visualization helpers ─────────────────────────────────────────────────

def shade_palette(hex_color: str, n: int = 4,
                  lo_lightness: float = 0.80,
                  hi_lightness: float = None) -> list:
    """Return n RGB tuples: lightest (lo_lightness) → darkest (hi_lightness)."""
    r, g, b = mcolors.to_rgb(hex_color)
    h, nat_l, s = colorsys.rgb_to_hls(r, g, b)
    hi = hi_lightness if hi_lightness is not None else max(nat_l, 0.25)
    lightnesses = np.linspace(lo_lightness, hi, n)
    return [colorsys.hls_to_rgb(h, float(lt), s) for lt in lightnesses]


def apply_y_format(ax, unit: str) -> None:
    if unit == '%':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: f'{x:.1f}%'))
    elif unit == 'days':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: f'{x:.0f}'))


def draw_boxes(ax, data_dict: dict, colors: list,
               highlight_baseline: bool = True) -> None:
    """Patch-colored box plots. data_dict: label → 1-D array."""
    labels = list(data_dict.keys())
    data   = [np.asarray(v, dtype=float) for v in data_dict.values()]
    data   = [v[~np.isnan(v)] for v in data]

    bp = ax.boxplot(
        data,
        labels=labels,
        whis=1.5,            # standard IQR whiskers (cleaner for expo)
        showfliers=False,
        patch_artist=True,
        widths=0.52,
        medianprops=dict(color='white', linewidth=3.0),   # white pops on colored boxes
        boxprops=dict(linewidth=1.3),
        whiskerprops=dict(linewidth=1.3, linestyle='--'),
        capprops=dict(linewidth=1.6),
    )
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.88)

    if highlight_baseline and len(data[0]) > 0:
        ax.axhline(np.median(data[0]),
                   color='#546E7A', linewidth=1.2,
                   linestyle=':', alpha=0.7)


GRAY_BASE = '#90A4AE'

print('Helpers defined.')

## Per-Group Box Plot Figures

One figure per scenario group (4 total), each with 6 metric subplots.  
Box = IQR · whiskers = 5th–95th pct across seeds · dotted line = baseline median.

In [ ]:
LTFU_ANNOTATION = (
    "↑ More intake patients compete for\n"
    "fixed procedure slots → longer queues\n"
    "→ higher LTFU"
)

for group_key, group in SCENARIO_GROUPS.items():
    shades     = shade_palette(group['color'], n=len(MULTIPLIERS))
    all_colors = [GRAY_BASE] + shades

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes_flat = axes.flatten()

    for ax_i, (metric_key, metric_label, unit) in enumerate(PLOT_METRICS):
        ax = axes_flat[ax_i]

        data_dict = {}

        # baseline
        data_dict['Baseline'] = baseline_df[metric_key].values if metric_key in baseline_df.columns else np.array([])

        # capacity levels
        for mult in MULTIPLIERS:
            pct  = int(round(mult * 100))
            name = f'{group_key.lower()}_{pct}pct'
            sdf  = scenario_dfs.get(name)
            arr  = sdf[metric_key].values if (sdf is not None and metric_key in sdf.columns) else np.array([])
            data_dict[f'{pct}%'] = arr

        draw_boxes(ax, data_dict, all_colors)
        apply_y_format(ax, unit)

        ax.set_title(metric_label, fontweight='bold', fontsize=14, pad=8)
        ax.set_xlabel('Capacity level', fontsize=12, labelpad=4)
        if unit in ('%', 'days'):
            ax.set_ylabel(unit, fontsize=12)
        ax.grid(True, alpha=0.22, axis='y', linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(axis='x', labelsize=11)
        ax.tick_params(axis='y', labelsize=11)

        # annotate the PCP primary-LTFU tradeoff
        if group_key == 'PCP' and metric_key == 'ltfu_rate_primary_pct':
            ax.annotate(
                LTFU_ANNOTATION,
                xy=(0.97, 0.97), xycoords='axes fraction',
                ha='right', va='top', fontsize=9,
                color='#B71C1C',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFEBEE',
                          edgecolor='#EF9A9A', alpha=0.9),
            )

    # figure-level legend
    legend_handles = [
        Patch(facecolor=GRAY_BASE, label='Baseline', alpha=0.88)
    ] + [
        Patch(facecolor=shades[j],
              label=f'{int(round(MULTIPLIERS[j]*100))}% of baseline',
              alpha=0.88)
        for j in range(len(MULTIPLIERS))
    ]
    fig.legend(
        handles=legend_handles,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.03),
        ncol=5,
        framealpha=0.95,
        fontsize=12,
    )

    fig.suptitle(
        f'Capacity Scenario Analysis: {group["label"]}\n'
        f'{group["subtitle"]}',
        fontsize=15, fontweight='bold', y=1.01,
    )
    plt.tight_layout(rect=[0, 0.04, 1, 1])

    out_path = OUTPUT_DIR / f'01_boxplot_{group_key.lower()}_capacity.png'
    fig.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'Saved → {out_path.name}')
    plt.show()

## Summary: Median % Change from Baseline

Line chart comparing **median % change** from baseline across all four scenario groups for each metric.  
Shows whether a given capacity lever moves the needle and in which direction.

In [ ]:
# ── compute median % change per scenario ──────────────────────────────────
baseline_medians = {
    mk: baseline_df[mk].dropna().median()
    for mk, *_ in PLOT_METRICS
}

summary_rows = []
for group_key, group in SCENARIO_GROUPS.items():
    for mult in MULTIPLIERS:
        pct  = int(round(mult * 100))
        name = f'{group_key.lower()}_{pct}pct'
        sdf  = scenario_dfs.get(name)
        row  = {'Group': group['label'], 'GroupKey': group_key,
                'Pct': pct, 'Multiplier': mult}
        for mk, ml, _ in PLOT_METRICS:
            b_med = baseline_medians.get(mk, 0)
            if sdf is not None and mk in sdf.columns and b_med != 0:
                med        = sdf[mk].dropna().median()
                row[mk] = 100.0 * (med - b_med) / abs(b_med)
            else:
                row[mk] = np.nan
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

# ── 2 × 3 line-chart grid — one panel per metric ─────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes_flat = axes.flatten()
x_ticks   = [int(round(m * 100)) for m in MULTIPLIERS]

for ax_i, (metric_key, metric_label, unit) in enumerate(PLOT_METRICS):
    ax = axes_flat[ax_i]

    for group_key, group in SCENARIO_GROUPS.items():
        sub = summary_df[summary_df['GroupKey'] == group_key]
        y_vals = sub.set_index('Pct')[metric_key].reindex(x_ticks).values

        ax.plot(
            x_ticks, y_vals,
            marker='o', linewidth=2.5, markersize=9,
            color=group['color'], label=group['label'],
            zorder=3,
        )

    ax.axhline(0, color='#455A64', linewidth=1.1, linestyle='--', alpha=0.55)
    ax.set_title(metric_label, fontweight='bold', fontsize=14, pad=8)
    ax.set_xlabel('Capacity level (% of baseline)', fontsize=12, labelpad=4)
    ax.set_ylabel('Median Δ from baseline (%)', fontsize=12)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'{p}%' for p in x_ticks], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(True, alpha=0.22, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# shared legend
legend_handles = [
    plt.Line2D([0], [0],
               color=g['color'], marker='o',
               linewidth=2.5, markersize=9,
               label=g['label'])
    for g in SCENARIO_GROUPS.values()
]
fig.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.03),
    ncol=4,
    fontsize=12,
    framealpha=0.95,
)
fig.suptitle(
    'Capacity Scenario Summary: Median % Change from Baseline\n'
    'by scenario group and metric',
    fontsize=15, fontweight='bold', y=1.01,
)
plt.tight_layout(rect=[0, 0.04, 1, 1])

out_path = OUTPUT_DIR / '05_summary_pct_change.png'
fig.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved → {out_path.name}')
plt.show()

In [ ]:
# ── tabular summary ───────────────────────────────────────────────────────
# Median % change for each group at 120% capacity (largest increment)
peak = summary_df[summary_df['Pct'] == 120].set_index('Group')
cols = [mk for mk, *_ in PLOT_METRICS]
display_peak = peak[cols].copy()
display_peak.columns = [ml for _, ml, _ in PLOT_METRICS]
display_peak = display_peak.round(2)
print('Median % change from baseline at 120% capacity:')
display_peak